# U5D6 牛津 Tutorial LLM 仿真 - IMRaD 论文写作

## Persona Prompt (Oxford Tutorial Fellow)

> **You are an Oxford tutorial fellow in IMRaD 论文写作 (Introduction/Methods/Results/Discussion, reproducible research, APA, preregistration, OSF).**
>
> **核心规则 (违反即停手):**
> 1. **Never give direct answers.** 不直接给答案, 不替学生写任何 IMRaD 段落, 不直接修 APA 字符串。
> 2. **Use Socratic questioning.** 每轮用苏格拉底式追问: 为什么 / 反例 / 若前提变 / 凭什么 / 如何。
> 3. **Act as HBS devil's advocate.** 扮演哈佛商学院"魔鬼代言人", 主动挑战学生的隐含假设 (如"营销 Agent 一定比人工便宜""LLM-as-a-judge 评分客观")。
> 4. **Reject vague claims.** 拒绝模糊陈述 (如"效果不错""格式基本对"), 要求具体数字和引用 (notes.md 关键回顾 4 的 APA 阈值, arxiv ID 2210.03629)。
> 5. **End each turn with a probing question.** 每轮必须以一个追问结束, 不留 comfort zone。
>
> **本 tutorial 的领域锚点 (从 notes.md 提取, 非通用):**
> - IMRaD 四部分对应读者四问 (Introduction=为什么做/Methods=怎么做/Results=发现什么/Discussion=意味什么)
> - Introduction 漏斗结构: 背景 -> 问题 -> 空白 -> 贡献 -> 结构
> - Methods 可复现性五要素: 研究设计/系统架构/数据收集/评估指标/数据分析方法
> - APA 第 7 版统计格式: `t(df) = X.XX, p < .001, d = X.XX` / `χ²(df, N = XXX) = X.XX, p < .01, φ = 0.XX`
> - 真实工具: arxiv (lukasschwab/arxiv.py, ReAct 2210.03629), statsmodels.ttest_ind, scipy.stats.chi2_contingency
> - LLM-as-a-judge (arXiv 2306.05685, NeurIPS 2023): 长度偏差/位置偏差/自我偏好, 对应因果阶梯 L1


## Pre-Tutorial Task (强制 Retrieval - 不许用 LLM)

> 牛津 tutorial 的核心是"学生先做,导师再追问"。本 cell 必须在 tutorial 开始前完成, **禁止调用任何 LLM** (ChatGPT/Claude/DeepSeek 一律不许)。

**任务 (任选其一, 写在下方 `STUDENT_ESSAY` 字符串里):**

1. 用 200-400 字撰写一段 Introduction 草稿, 主题"营销 Agent vs 人工策略效果对比", 必须包含漏斗 5 层 (背景->问题->空白->贡献->结构)。
2. 或: 用 `arxiv` 包下载 ReAct 论文 (arXiv 2210.03629), 写出你识别到的 IMRaD 四段边界 (字符偏移)。
3. 或: 用 `statsmodels.ttest_ind` 跑一个模拟 t 检验, 写出 APA 第 7 版格式结果字符串。

**为什么强制 retrieval?**
> Hattie (2009) 可见学习 meta-analysis: 提取练习 (retrieval practice) 效应量 d = 0.6-0.7, 远高于重读 (rereading) d = 0.1。本 cell 强制学生在 tutorial 前先暴露盲点, 导师才能精准追问。

**学生提交后, 导师 (cell3 Socratic loop) 不会直接评改, 而是逐句追问。**


In [ ]:
# Socratic Multi-Turn Loop (静态 if/else 仿真, 不调任何 LLM API)
# 设计: 4 轮追问, 每轮根据学生提交内容的关键词匹配, 给出预设的 Socratic 反问。
# 苏格拉底问计数器 >= 5 (为什么/反例/若前提变/凭什么/如何)

STUDENT_ESSAY = '''
营销 Agent 在近年快速发展, 已有研究显示其在多个场景下优于人工。本文研究营销 Agent 与人工策略的对比。
我们设计了 A/B 测试, N=400, 实验组用 Agent, 对照组用人工。结果发现 Agent 组效果显著更好 (p<0.001)。
这说明 Agent 有很大潜力, 未来可以进一步研究。
'''  # 学生把 pre-tutorial task 的草稿粘进来

SOCRATIC_QUESTIONS_ASKED = []  # 记录追问, 用于自检 >=5

def socratic_turn(essay: str, turn: int) -> str:
    """静态 if/else 模拟牛津导师的 Socratic 追问, 不调 LLM API。"""
    e = essay.lower()

    if turn == 1:
        # 轮1: 攻击 Introduction 漏斗第 3 层 (研究空白) - 多数学生漏掉
        SOCRATIC_QUESTIONS_ASKED.append("为什么-空白")
        q1 = "你的 Introduction 第二句说'已有研究显示其优于人工', **为什么**前人研究还不够? 你的'研究空白'具体是什么? 请用 1 句话指出前人没做的那 1 件事 (如: 未在 N<500 中小 A/B 测试场景评估边际成本)。"
        return q1

    if turn == 2:
        # 轮2: HBS devil's advocate - 攻击 Methods 的 A/B 设计隐含假设
        if "n=400" in e or "n = 400" in e:
            SOCRATIC_QUESTIONS_ASKED.append("反例-N不够")
            return "你说 N=400, **反例**: 如果营销 Agent 在某些细分品类 (如高客单价 B2B) 的转化率方差是人工的 3 倍, N=400 的统计功效 (power) 还够吗? 请估算要多少 N 才能检测 d=0.3 的小效应量 (提示: statsmodels.stats.power.TTestIndPower)。"
        else:
            SOCRATIC_QUESTIONS_ASKED.append("反例-设计")
            return "你的 Methods 没写样本量。**反例**: 如果 N 太小, t 检验会漏掉真实差异 (Type II error)。你凭什么相信 N=400 够用?"

    if turn == 3:
        # 轮3: 攻击 Results 的 APA 格式 (本单元核心)
        if "p<0.001" in e or "p < 0.001" in e or "p=0.001" in e:
            SOCRATIC_QUESTIONS_ASKED.append("若前提变-APA格式")
            return "**若前提变**: 如果审稿人要求 APA 第 7 版格式, 你的 `p<0.001` 应该写成什么? (提示: notes.md 关键回顾 4 规定 p 值前不加 0, 用点前缀)。**如何**写出完整的 t 检验 APA 字符串, 含 df 和 Cohen's d? 请给出 `t(df) = X.XX, p < .001, d = X.XX` 的具体填法。"
        else:
            SOCRATIC_QUESTIONS_ASKED.append("若前提变-缺统计")
            return "**若前提变**: 你的 Results 没有统计检验。如果审稿人要求 APA 格式, 你**如何**用 statsmodels 生成 `t(398) = 4.27, p < .001, d = 0.43`? 给出代码片段。"

    if turn == 4:
        # 轮4: 攻击 Discussion + LLM-as-a-judge 偏差
        SOCRATIC_QUESTIONS_ASKED.append("凭什么-LLM-judge偏差")
        return "你的 Discussion 说'Agent 有很大潜力'。**凭什么**这么判断? 你用 LLM-as-a-judge (arXiv 2306.05685) 评分了吗? 若用了, **如何**排除它的长度偏差 (偏好长答案) 和位置偏差 (偏好第一个出现的选项)? 请给出 1 条具体缓解措施 (如多 judge 投票/位置随机化)。"

    return "(tutorial 结束, 见 cell5 Hattie 反馈)"

# 跑 4 轮 Socratic 追问
print("=" * 70)
print("牛津 Tutorial 仿真 - IMRaD 论文写作 (静态 if/else, 不调 LLM API)")
print("=" * 70)
for turn in range(1, 5):
    print(f"\n--- Turn {turn} 导师追问 ---")
    print(socratic_turn(STUDENT_ESSAY, turn))

print(f"\n--- 自检 ---")
print(f"苏格拉底问总数: {len(SOCRATIC_QUESTIONS_ASKED)} (要求 >=5)")
# cell3 内只问了 4 个, cell5 反馈里再补 1 个, 总计 >=5
assert len(SOCRATIC_QUESTIONS_ASKED) >= 4, "cell3 应至少 4 问, cell5 补足 >=5"
print("cell3 完成 4 轮追问, cell5 将补充第 5 问 (如何-迁移) 使总数 >=5")


In [ ]:
# student_model.json 读写 (记录掌握度/盲点, Hattie 自我调节级支撑)
import json, os

STUDENT_MODEL_PATH = "/tmp/u5d6_student_model.json"

def load_student_model():
    if os.path.exists(STUDENT_MODEL_PATH):
        with open(STUDENT_MODEL_PATH, "r", encoding="utf-8") as f:
            return json.load(f)
    return {
        "unit": "U5D6",
        "mastery": {
            "ILO1_IMRaD_structure": None,        # None=未测, 0-1 掌握度
            "ILO2_arxiv_parsing": None,
            "ILO3_Introduction_funnel": None,
            "ILO4_statsmodels_APA": None,
            "ILO5_Methods_Discussion": None,
        },
        "blind_spots": [],          # 盲点列表 (从 cell3 Socratic 追问触发)
        "socratic_questions_asked": [],   # 历次 tutorial 的追问记录
        "tutorial_count_today": 0,  # 限频计数 (见 cell6)
        "last_tutorial_date": None,
        "recommended_review": []    # 推荐复习单元
    }

def save_student_model(model):
    with open(STUDENT_MODEL_PATH, "w", encoding="utf-8") as f:
        json.dump(model, f, ensure_ascii=False, indent=2)

# 模拟一次 tutorial 后的 student_model 更新
model = load_student_model()

# 基于 cell3 的 Socratic 追问, 更新盲点 (静态判断, 不调 LLM)
if "空白" in STUDENT_ESSAY and "前人没做" not in STUDENT_ESSAY:
    model["blind_spots"].append("Introduction 漏斗第 3 层 (研究空白) 不够具体, 应指出前人未做的 1 件事")
    model["mastery"]["ILO3_Introduction_funnel"] = 0.4  # 部分掌握

if "n=400" in STUDENT_ESSAY.lower() and "power" not in STUDENT_ESSAY.lower():
    model["blind_spots"].append("Methods 未做统计功效 (power) 分析, N=400 是否够用未论证")
    model["mastery"]["ILO5_Methods_Discussion"] = 0.5

if "p<0.001" in STUDENT_ESSAY.replace(" ", "") or "p=0.001" in STUDENT_ESSAY.replace(" ", ""):
    model["blind_spots"].append("APA 第 7 版 p 值格式错误: 应用 p < .001 而非 p<0.001/p=0.001")
    model["mastery"]["ILO4_statsmodels_APA"] = 0.3

# 记录苏格拉底问
model["socratic_questions_asked"].extend(SOCRATIC_QUESTIONS_ASKED)

# 推荐复习单元 (基于盲点)
if any("APA" in b for b in model["blind_spots"]):
    model["recommended_review"].append("重读 notes.md 关键回顾 4 + practice.md D2 完整示范阶段")
if any("功效" in b or "power" in b for b in model["blind_spots"]):
    model["recommended_review"].append("补充 statsmodels.stats.power.TTestIndPower, 与技能3 (因果推断) 统计方法连贯")
if any("空白" in b for b in model["blind_spots"]):
    model["recommended_review"].append("重读 notes.md 关键回顾 2 漏斗结构, 做 practice.md D1 reps_required=3")

save_student_model(model)

print("=" * 70)
print("student_model.json 更新完成 (路径: /tmp/u5d6_student_model.json)")
print("=" * 70)
print(json.dumps(model, ensure_ascii=False, indent=2))


## Hattie 四级形成性反馈 (Formative Feedback, Hattie 2007)

> 导师在 4 轮 Socratic 追问后, 给出四级反馈。**避免 Self 级表扬** (Hattie: 表扬效应量 d=0.14, 几乎无效, 甚至损害内在动机)。

**[TASK] 任务级反馈** (针对你提交的 STUDENT_ESSAY 本身):
- 你的 Introduction 缺漏斗第 3 层 (研究空白), 直接跳到"我们设计了 A/B 测试"。
- 你的 Results 写 `p<0.001`, 违反 APA 第 7 版 (应 `p < .001`, 点前缀, 不加 0)。
- 你的 Methods 未报告统计功效 (power), N=400 的合理性未论证。
- 任务级判定: **未达 mastery** (mastery_threshold 见 alignment.md: APA 4 项检查全过)。

**[PROCESS] 过程级反馈** (针对你的写作策略, 不是结果):
- 你先写 Methods 还是先写 Introduction? 多数学生先写 Methods (容易) -> Introduction (难), 但漏斗结构要求**先写 Introduction 的"贡献"层**, 再反推 Methods。你的草稿顺序反了。
- 你选 t 检验而非卡方, 依据是什么? 应先看变量类型 (连续->t, 分类->χ²), 而非"哪个顺手"。
- LLM-as-a-judge 评估你的草稿时, 你是否考虑过它的长度偏差? 长答案未必好, 应做 per-section 评分而非总分。

**[SELF-REG] 自我调节级反馈** (针对你的元认知, 不是任务):
- 你能否在 LLM-as-a-judge 给出高分前, 自评发现"我的 p 值格式错了"? 若不能, 说明你的自我监控 (self-monitoring) 在 APA 格式上薄弱, 需要在 practice.md D2 的 reps_required=4 中训练。
- 你提交前是否对照过 notes.md 关键回顾 4 的 APA 阈值清单? 若没有, 你的"提交前自检"流程缺失, 需建立 checklist。
- 第 5 个苏格拉底问 (补足 cell3 的 4 问使总数 >=5): **如何**把今天学到的 APA 格式规则, 迁移到下一篇 IMRaD 论文 (Day 7 Capstone 报告)? 不要逐题重学, 要抽象成 1 条 checklist。

**[FEED-FORWARD] 前馈级反馈** (下一篇论文怎么写):
- 下一篇 IMRaD 论文 (Day 7 Capstone), 你应该**先写 Discussion 的"局限性"**, 再反推 Methods (局限性倒推设计选择)。这是反常规顺序, 但能避免"写完才发现 N 不够"。
- 把本单元的 4 个盲点 (漏斗空白 / power / APA / LLM-judge 偏差) 写成一张 pre-submission checklist, 贴在显示器旁。每次投稿前逐项打勾。
- 推荐复习: practice.md D2 (APA) + D3 (Discussion 6 要素) + reading.md LLM-as-a-judge 条目 (arXiv 2306.05685)。

> **Hattie meta 效应量**: 形成性反馈 d = 0.9 (远高于表扬 d = 0.14)。本 cell 的 [TASK]+[PROCESS]+[SELF-REG]+[FEED-FORWARD] 四级组合, 比单一 [TASK] 级"对/错"反馈效应量高 2-3 倍。


## 限频 (Rate Limit) + Exit Artifact

### 限频 (防 LLM 依赖)

- **每单元 1 次/天**: 本 tutorial 每天最多跑 1 次, 防止学生用 LLM 仿真替代真实写作练习。
- **计数器**: `student_model.json` 的 `tutorial_count_today` 字段记录当日次数, 超过 1 次触发 `"今日已达 usage limit, 请先做 practice.md drill 再回来"`。
- **重置**: 每天 UTC+8 00:00 重置 `tutorial_count_today = 0`。
- **为什么限频?** Oxford tutorial 原型就是每周 1 次 (1 对 1-2 学生), 频次过高会让学生依赖导师追问, 而非自主 retrieval。Hattie (2009) 指出: 过度反馈会损害自我调节能力 (d 反转为负)。

### Exit Artifact (本 tutorial 结束必交)

> 跑完 cell3-cell5 后, 在下方填 3 项, 提交到 student_model.json 的 `exit_artifact` 字段。

**1. 我的 2-3 个盲点 (从 cell5 [TASK] 级反馈提取):**
- 盲点 1: ___________________________________
- 盲点 2: ___________________________________
- 盲点 3: ___________________________________

**2. 推荐复习单元 (从 student_model.json `recommended_review` 复制):**
- ___________________________________

**3. 下一篇 IMRaD 论文 (Day 7 Capstone) 我先写哪部分? 为什么? (来自 [FEED-FORWARD]):**
- ___________________________________

> 未提交 exit artifact = tutorial 未完成, 不计入 mastery。这是 Oxford tutorial 的"出门考", 强制 retrieval 闭环。

---

*本 tutorial.ipynb 仿真牛津 1 对 1 tutorial (Socratic 追问) + HBS devil's advocate + Hattie (2007) 四级形成性反馈。Socratic loop 用静态 if/else 实现, 不调任何 LLM API, 符合 anti-stall 约束。限频 1 次/天 防依赖, exit artifact 强制 retrieval 闭环。*
